In [1]:
## Scenic Env ##

import os
import glob
import gc
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

# pySCENIC imports

from dask.distributed import Client
from dask.diagnostics import ProgressBar

from arboreto.algo import grnboost2

from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
from pyscenic.rss import regulon_specificity_scores

import networkx as nx
import community as community_louvain
from collections import defaultdict

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase

%matplotlib inline

/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/l

In [2]:
# DEFINE PATHS ##

SCENIC_PATH = '/mnt/sdb/scz_meta_analysis_processed/scenic_files/'
FEATHER_RANKINGS_FILE = 'hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
DATABASES_GLOB = os.path.join(SCENIC_PATH, FEATHER_RANKINGS_FILE)
MOTIF_ANNOTATIONS_FNAME = os.path.join(SCENIC_PATH, 'motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl')

In [3]:
## SAVE LOAD ##

# # Organoids #
ORGANOIDS_PATH = '/mnt/sdb/scz_meta_analysis_processed/anndata_objs'
organoids_filename = 'integrated_adata_annotated_w_celltypist_scanvi.h5ad'
scz_adata = sc.read_h5ad(os.path.join(ORGANOIDS_PATH, organoids_filename), backed=True)

# TF List ##
TF_LIST = "/home/deepak/datasets/developing_hippocampus/roussos_organoid_snrnaseq/scenic_files/allTFs_hg38.txt"
tf_list = pd.read_csv(TF_LIST, header=None)[0].tolist()

In [4]:
# drop doublets #
droplet_celltype_bool = ((~scz_adata.obs['pred_dbl']) & (scz_adata.obs['subclass_annotations_markers'] != 'Stressed Glia FTL+B2M+GLUL+'))
gc.collect()

29081

In [5]:
# Manuscripts #

# define manuscripts #
sebastian = scz_adata[(scz_adata.obs['Manuscript'] == 'Sebastian') & droplet_celltype_bool].to_memory()
rao = scz_adata[(scz_adata.obs['Manuscript'] == 'Rao') & droplet_celltype_bool].to_memory()
fernando = scz_adata[(scz_adata.obs['Manuscript'] == 'Fernando') & droplet_celltype_bool].to_memory()
walsh = scz_adata[(scz_adata.obs['Manuscript'] == 'Walsh') & droplet_celltype_bool].to_memory()
notaras = scz_adata[(scz_adata.obs['Manuscript'] == 'Notaras') & droplet_celltype_bool].to_memory()
purcell = scz_adata[(scz_adata.obs['Manuscript'] == 'Purcell') & droplet_celltype_bool].to_memory()
khan = scz_adata[(scz_adata.obs['Manuscript'] == 'Khan') & droplet_celltype_bool].to_memory()
shin = scz_adata[(scz_adata.obs['Manuscript'] == 'Shin') & droplet_celltype_bool].to_memory()
sawada = scz_adata[(scz_adata.obs['Manuscript'] == 'Sawada') & droplet_celltype_bool].to_memory()

# form manuscript dictionary #
manuscripts = {'sebastian': sebastian, 'fernando': fernando, 'rao': rao, 'walsh':walsh, 'notaras':notaras, 
               'purcell':purcell, 'khan':khan, 'shin':shin, 'sawada':sawada}

In [6]:
# Build Counts Matrices

counts_matrices = {}

for name, manuscript in manuscripts.items():

    print(f"Processing {name}")

    # Work with raw counts
    manuscript.X = manuscript.layers["raw_counts"]

    # Define donor/sample/celltype identifier
    manuscript.obs["Donor_Sample_Celltype"] = (manuscript.obs["Donor"].astype(str)
        + "_" + manuscript.obs["Sample Name"].astype(str)
        + "_" + manuscript.obs["CellType"].astype(str))

    # Compute HVGs on normalized data
    sc.pp.highly_variable_genes(
        manuscript,
        layer="log1p_norm",
        n_top_genes=10000,
        subset=False,
        inplace=True)

    # Keep TFs expressed in >=1% of cells (minimum 10 cells)
    min_cells = max(10, int(0.01 * manuscript.n_obs))

    expressed_TFs = manuscript.var_names[
        manuscript.var_names.isin(tf_list)
        & ((manuscript.X > 0).sum(axis=0).A1 >= min_cells)
    ]

    # Union of HVGs and expressed TFs
    gene_filter = (manuscript.var["highly_variable"] | manuscript.var_names.isin(expressed_TFs))

    manuscript = manuscript[:, gene_filter].copy()
    gc.collect()

    print(f"Retained {manuscript.n_vars:,} genes")

    # Cell-level counts (genes × cells)
    counts_matrix_not_pb = pd.DataFrame(
        manuscript.X.toarray().T,
        index=manuscript.var["hgnc_symbol"],
        columns=manuscript.obs_names,
    )

    # Pseudobulk by donor/sample/celltype
    counts_matrices[name] = (
        counts_matrix_not_pb
        .groupby(manuscript.obs["Donor_Sample_Celltype"], axis=1)
        .agg(np.nanmean)
    )

    del counts_matrix_not_pb
    gc.collect()

Processing sebastian


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


Retained 11,760 genes


/tmp/ipykernel_327714/1770070296.py:50: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  counts_matrix_not_pb
/tmp/ipykernel_327714/1770070296.py:52: FutureWarning: The provided callable <function nanmean at 0x7fa8b829b6d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.nanmean)


Processing fernando


...

/tmp/ipykernel_327714/1770070296.py:50: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  counts_matrix_not_pb
/tmp/ipykernel_327714/1770070296.py:52: FutureWarning: The provided callable <function nanmean at 0x7fa8b829b6d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.nanmean)


Processing sawada


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


Retained 15,622 genes


/tmp/ipykernel_327714/1770070296.py:50: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  counts_matrix_not_pb
/tmp/ipykernel_327714/1770070296.py:52: FutureWarning: The provided callable <function nanmean at 0x7fa8b829b6d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.nanmean)


In [8]:
## Run GRNBoost2 ##

client = Client(n_workers=1, threads_per_worker=24, processes=False, dashboard_address=None)

adjacencies = {}

for name, counts_matrix in counts_matrices.items():

    expr = counts_matrix.T
    expr.columns = expr.columns.astype(str)

    adjacencies[name] = grnboost2(expression_data=expr, gene_names=expr.columns,
                                  tf_names=list(expr.columns[expr.columns.isin(tf_list)]),
                                  client_or_address=client, verbose=True)

preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/distributed/client.py:3106: UserWarning: Sending large graph of size 26.85 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


not shutting down client, client was created externally
finished
preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/distributed/client.py:3106: UserWarning: Sending large graph of size 14.01 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


not shutting down client, client was created externally
finished
preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


...

/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/distributed/client.py:3106: UserWarning: Sending large graph of size 16.59 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


not shutting down client, client was created externally
finished
preparing dask client
parsing input
creating dask graph
1 partitions
computing dask graph


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/distributed/client.py:3106: UserWarning: Sending large graph of size 11.42 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


not shutting down client, client was created externally
finished


2026-08-09 16:02:33,477 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 352.55 GiB -- Worker memory limit: 503.56 GiB
2026-08-09 16:02:43,576 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 355.41 GiB -- Worker memory limit: 503.56 GiB
2026-08-09 16:02:53,676 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more informati

In [9]:
for manuscript in adjacencies.keys():
    adjacencies[manuscript].to_csv(f"/mnt/sdb/scz_meta_analysis_processed/scenic_files/grnboost2_{manuscript}_adjacencies.tsv",
                                     index=False)

In [10]:
## Load Adjacencies To Avoid Recalculating ##

adjacencies = {}

for manuscript in manuscripts.keys():
    adjacencies[manuscript] = pd.read_csv(f"{SCENIC_PATH}/grnboost2_{manuscript}_adjacencies.tsv", sep=",")

In [11]:
# ## Regulon prediction aka cisTarget from CLI ##

# Load ranking databases once
db_fnames = glob.glob(DATABASES_GLOB)

def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]

dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]

# Store outputs
all_modules = {}
all_motif_dfs = {}
all_regulons = {}

for manuscript_name, adjacency in adjacencies.items():

    print(f"Processing {manuscript_name}")

    # expression matrix corresponding to this manuscript
    expr = counts_matrices[manuscript_name].T

    # Build co-expression modules
    modules = list(modules_from_adjacencies(adjacency, expr))

    all_modules[manuscript_name] = modules

    # cisTarget motif enrichment/pruning
    with ProgressBar():
        df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME)

    all_motif_dfs[manuscript_name] = df

    # Convert enriched motifs into regulons
    regulons = df2regulons(df)

    all_regulons[manuscript_name] = regulons

    # Save regulons immediately
    with open(f"{SCENIC_PATH}/{manuscript_name}_regulons.pkl", "wb") as f:
        pickle.dump(regulons, f)


2026-08-09 15:38:22,373 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-08-09 15:38:22,477 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].


Processing sebastian



2026-08-09 15:38:30,007 - pyscenic.utils - INFO - Creating modules.


[                                        ] | 0% Completed | 114.55 ms

/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/dask/base.py:1368: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(


...

[##################################      ] | 86% Completed | 125.71 s


2026-08-09 16:01:56,367 - pyscenic.transform - WARNING - Less than 80% of the genes in PTF1A could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.


[##################################      ] | 86% Completed | 128.53 s


2026-08-09 16:01:59,227 - pyscenic.transform - WARNING - Less than 80% of the genes in ZNF669 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings. Skipping this module.


[########################################] | 100% Completed | 136.51 s
Create regulons from a dataframe of enriched features.
Additional columns saved: []


In [12]:
# Reload Regulons #

all_regulons = {}

for manuscript in manuscripts.keys():

    with open(f"{SCENIC_PATH}/{manuscript}_regulons.pkl", "rb") as f:
        all_regulons[manuscript] = pickle.load(f)

In [ ]:
all_auc = {}

for manuscript_name, regulons in all_regulons.items():

    print(f"Scoring {manuscript_name}")

    adata = manuscripts[manuscript_name]

    print('building expr')
    expr = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    
    print('building auc_mtx')
    auc_mtx = aucell(expr, regulons, num_workers=4)

    print('add to dictionary')
    all_auc[manuscript_name] = auc_mtx

    del expr
    gc.collect()

Scoring sebastian
building expr
building auc_mtx


In [ ]:
## Save AUCs ##

output_file = os.path.join(SCENIC_PATH, "all_manuscript_auc.pkl")

with open(output_file, "wb") as f:
    pickle.dump(all_auc, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved AUC matrices to {output_file}")